# ECharts Notebook Smoke Test

This notebook verifies that Apache ECharts can render from a Jupyter notebook using `pyecharts`. It intentionally stays independent of `fastcharts` so we can use it as a clean comparison/control chart when testing notebook behavior.

In [ ]:
try:
    from IPython.display import display
    from pyecharts import options as opts
    from pyecharts.charts import Bar, Line
    from pyecharts.globals import CurrentConfig
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Install notebook example dependencies with: "
        "uv pip install nbformat nbclient nbconvert pyecharts"
    ) from exc

# The default pyecharts asset host can be sensitive to certificate/date
# issues in automated browsers. Use a version-pinned public ECharts build.
CurrentConfig.ONLINE_HOST = "https://cdn.jsdelivr.net/npm/echarts@5.6.0/dist/"

print("pyecharts import ok")

## Combined Bar + Line Chart

The chart below exercises the ECharts runtime, tooltip config, dual y-axes, and an overlapped series.

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
revenue = [42, 58, 61, 73, 86, 97]
latency_ms = [38, 34, 31, 29, 25, 22]

bar = (
    Bar(init_opts=opts.InitOpts(width="900px", height="420px", renderer="canvas"))
    .add_xaxis(months)
    .add_yaxis("Revenue", revenue, color="#3b82f6")
    .extend_axis(
        yaxis=opts.AxisOpts(
            name="Latency ms",
            position="right",
            axislabel_opts=opts.LabelOpts(formatter="{value} ms"),
        )
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(title="ECharts notebook smoke test"),
        tooltip_opts=opts.TooltipOpts(trigger="axis"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
        xaxis_opts=opts.AxisOpts(name="Month"),
        yaxis_opts=opts.AxisOpts(name="Revenue"),
    )
)

line = Line().add_xaxis(months).add_yaxis("Latency ms", latency_ms, yaxis_index=1, color="#ef4444")

chart = bar.overlap(line)
notebook_chart = chart.render_notebook()
html = notebook_chart.data

assert "cdn.jsdelivr.net/npm/echarts@5.6.0/dist/echarts.min" in html
assert "require(['echarts']" in html
assert "echarts.init" in html
assert "setOption" in html
assert "ECharts notebook smoke test" in html
print("ECharts embed smoke test passed")

display(notebook_chart)